### Natural Language Processing (NLP) 🧠📝

In [1]:
# Check for GPU  
!nvidia-smi

Sun Mar  8 17:13:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!wget https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/refs/heads/main/extras/helper_functions.py

--2026-03-08 17:13:48--  https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/refs/heads/main/extras/helper_functions.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10246 (10K) [text/plain]
Saving to: ‘helper_functions.py.1’

helper_functions.py 100%[===================>]  10.01K  --.-KB/s    in 0s      

2026-03-08 17:13:48 (121 MB/s) - ‘helper_functions.py.1’ saved [10246/10246]



In [5]:
from helper_functions import unzip_data , create_tensorboard_callback , plot_loss_curves , compare_historys

## Get a text database

The database we're going to be using is Kaggle's intruduction to NLP dataset (text samples of Tweets labelled as disaster or not disaster)

In [3]:
!wget https://storage.googleapis.com/ztm_tf_course/nlp_getting_started.zip

# Unzip data 
unzip_data("nlp_getting_started.zip")

--2026-03-08 17:13:12--  https://storage.googleapis.com/ztm_tf_course/nlp_getting_started.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 172.253.118.207, 74.125.200.207, 74.125.130.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|172.253.118.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 607343 (593K) [application/zip]
Saving to: ‘nlp_getting_started.zip’

nlp_getting_started 100%[===================>] 593.11K   720KB/s    in 0.8s    

2026-03-08 17:13:13 (720 KB/s) - ‘nlp_getting_started.zip’ saved [607343/607343]



##  Visualizing a text dataset 

In [6]:
import pandas as pd 

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

train_df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [7]:
train_df["text"][0] 

'Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all'

In [8]:
# Shuffle training dataframe 
train_df_shuffled = train_df.sample(frac=1 , random_state=42)
train_df_shuffled.head()

,id,keyword,location,text,target
2644,3796,destruction,NaN,So you have a new weapon that can cause un-ima...,1
2227,3185,deluge,NaN,The f$&amp;@ing things I do for #GISHWHES Just...,0
5448,7769,police,UK,DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe...,1
132,191,aftershock,NaN,Aftershock back to school kick off was great. ...,0
6845,9810,trauma,"Montgomery County, MD",in response to trauma Children of Addicts deve...,0


In [9]:
# What does the test dataframe look like 
test_df.head()

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


In [11]:
# Examples of each class
train_df.target.value_counts()

,count
target,
0,4342
1,3271


In [10]:
# How many total samples 
len(train_df) , len(test_df)

(7613, 3263)

In [12]:
# Let's visualize some random training examples 
import random 

random_index = random.randint(0, len(train_df)-5)  # create random indexes not higher than the total number of samples  
for row in train_df_shuffled[["text" , "target"]][random_index:random_index+5].itertuples():
    print(f"Text: {row.text}")
    print(f"Target: {row.target}")
    print("---")

Text: holy crap @KingMyth1999 my phone just exploded. haha
Target: 0
---
Text: @zourryart I forgot to add the burning buildings and screaming babies
Target: 1
---
Text: Breaking news: Haunting memories drawn by survivors http://t.co/PCjBvrs7xw
Target: 1
---
Text: Reminder: Mass murderer and white supremacist Anders Breivik was also unsurprisingly an anti-feminist.
http://t.co/1lXnJVl8TR
Target: 1
---
Text: do he love me do he love me not I ain't a playa I just crush a lot
Target: 0
---


In [15]:
from sklearn.model_selection import train_test_split

In [16]:
# Split the data
train_sentence , val_sentence , train_labels , val_labels = train_test_split(train_df_shuffled["text"].to_numpy() , train_df_shuffled["target"].to_numpy() , test_size=0.1 , random_state=42)

In [17]:
# Check the lengths 
len(train_sentence) , len(train_labels) , len(val_sentence) , len(val_labels)

(6851, 6851, 762, 762)

In [18]:
len(train_df_shuffled)

7613

In [19]:
# Check some examples 
train_sentence[:10] , train_labels[:10]

(array(['@mogacola @zamtriossu i screamed after hitting tweet',
        'Imagine getting flattened by Kurt Zouma',
        '@Gurmeetramrahim #MSGDoing111WelfareWorks Green S welfare force ke appx 65000 members har time disaster victim ki help ke liye tyar hai....',
        "@shakjn @C7 @Magnums im shaking in fear he's gonna hack the planet",
        'Somehow find you and I collide http://t.co/Ee8RpOahPk',
        '@EvaHanderek @MarleyKnysh great times until the bus driver held us hostage in the mall parking lot lmfao',
        'destroy the free fandom honestly',
        'Weapons stolen from National Guard Armory in New Albany still missing #Gunsense http://t.co/lKNU8902JE',
        '@wfaaweather Pete when will the heat wave pass? Is it really going to be mid month? Frisco Boy Scouts have a canoe trip in Okla.',
        'Patient-reported outcomes in long-term survivors of metastatic colorectal cancer - British Journal of Surgery http://t.co/5Yl4DC1Tqt'],
       dtype=object),
 array([0,